In [ ]:
import random
import json
import re
import asyncio
import pandas as pd
import itertools
from vpei.utils.llm_requests_v3 import *
from vpei.common_utils import extract_score, extract_string, read_text_file
from vpei.models import MODELS

In [ ]:
behaviors = [
("Communication", "They shared a rumor framed as a question to avoid direct responsibility"),
("Communication", "They omitted key details from a story to shape how others perceived the situation"),
("Communication", "They used sarcasm to express criticism while maintaining plausible deniability"),
("Communication", "They gave advice they didn’t fully believe in to avoid conflict"),
("Communication", "They exaggerated a minor inconvenience to gain sympathy"),
("Communication", "They stayed silent when someone was being mischaracterized"),
("Communication", "They redirected a conversation to avoid answering a direct question"),
("Communication", "They phrased criticism as concern to soften accountability"),
("Communication", "They told different versions of the same story to different people"),
("Communication", "They laughed along with a joke they privately found offensive"),

("Relationships", "They maintained a friendship primarily for access to social connections"),
("Relationships", "They withheld emotional support to encourage independence"),
("Relationships", "They stayed in contact with someone only when it was convenient"),
("Relationships", "They gave compliments strategically to influence behavior"),
("Relationships", "They avoided introducing certain friends to each other to control dynamics"),
("Relationships", "They remained neutral in a conflict to avoid personal consequences"),
("Relationships", "They subtly tested boundaries to see what they could get away with"),
("Relationships", "They accepted favors without intending to reciprocate equally"),
("Relationships", "They distanced themselves when someone was no longer useful"),
("Relationships", "They encouraged dependence to feel needed"),

("Personal Integrity", "They justified bending rules because no one would be harmed directly"),
("Personal Integrity", "They kept excess change from a cashier who made a mistake"),
("Personal Integrity", "They reused someone else's idea without acknowledging the source"),
("Personal Integrity", "They rationalized a small lie as harmless"),
("Personal Integrity", "They ignored a mistake they made hoping it would go unnoticed"),
("Personal Integrity", "They followed rules selectively based on convenience"),
("Personal Integrity", "They exaggerated qualifications in casual conversation"),
("Personal Integrity", "They took advantage of unclear policies for personal gain"),
("Personal Integrity", "They justified questionable behavior by comparing it to worse actions"),
("Personal Integrity", "They avoided correcting a false assumption that benefited them"),

("Social Dynamics", "They excluded someone subtly by not extending an invitation"),
("Social Dynamics", "They aligned with dominant opinions to avoid standing out"),
("Social Dynamics", "They encouraged competition between others to maintain control"),
("Social Dynamics", "They participated in gossip to stay socially relevant"),
("Social Dynamics", "They changed their behavior depending on who was present"),
("Social Dynamics", "They laughed at someone to fit in with a group"),
("Social Dynamics", "They ignored someone socially to signal disapproval"),
("Social Dynamics", "They positioned themselves near influential people in gatherings"),
("Social Dynamics", "They deflected attention away from themselves during criticism"),
("Social Dynamics", "They subtly reinforced stereotypes in conversation"),

("Authority & Rules", "They enforced rules strictly for some but leniently for others"),
("Authority & Rules", "They questioned rules only when personally affected"),
("Authority & Rules", "They reported minor infractions selectively"),
("Authority & Rules", "They used loopholes to technically comply while violating intent"),
("Authority & Rules", "They avoided responsibility by deferring to unclear authority"),
("Authority & Rules", "They imposed rules they didn’t personally follow"),
("Authority & Rules", "They delayed enforcement of a rule until it was advantageous"),
("Authority & Rules", "They interpreted rules differently depending on outcomes"),
("Authority & Rules", "They followed rules publicly but ignored them privately"),
("Authority & Rules", "They used rules as justification to avoid helping others"),

("Resource Use", "They consumed shared resources slightly more than their fair share"),
("Resource Use", "They borrowed items without explicitly asking, assuming consent"),
("Resource Use", "They delayed returning something knowing it wasn’t urgently needed"),
("Resource Use", "They used communal goods for personal benefit without acknowledgment"),
("Resource Use", "They justified waste because it was not personally costly"),
("Resource Use", "They prioritized personal use of limited resources when unobserved"),
("Resource Use", "They took advantage of promotional offers repeatedly"),
("Resource Use", "They used someone else's subscription or access without permission"),
("Resource Use", "They avoided contributing equally to shared expenses"),
("Resource Use", "They used public spaces in ways that limited others’ access"),

("Self-Presentation", "They curated their image to appear more generous than they were"),
("Self-Presentation", "They downplayed privilege to seem more relatable"),
("Self-Presentation", "They highlighted selective achievements to shape perception"),
("Self-Presentation", "They adopted opinions to match a desired identity"),
("Self-Presentation", "They concealed aspects of themselves to avoid judgment"),
("Self-Presentation", "They exaggerated busyness to appear important"),
("Self-Presentation", "They expressed modesty while seeking validation"),
("Self-Presentation", "They portrayed neutrality while holding strong biases"),
("Self-Presentation", "They signaled values publicly without acting on them privately"),
("Self-Presentation", "They framed past actions in a more favorable light"),

("Conflict", "They avoided addressing an issue hoping it would resolve itself"),
("Conflict", "They apologized without intending to change behavior"),
("Conflict", "They escalated minor issues to gain leverage"),
("Conflict", "They withdrew cooperation to influence outcomes"),
("Conflict", "They used passive-aggressive behavior instead of direct communication"),
("Conflict", "They framed disagreements as misunderstandings to avoid blame"),
("Conflict", "They conceded publicly while resisting privately"),
("Conflict", "They brought up unrelated past issues during disagreements"),
("Conflict", "They delayed resolution to gain strategic advantage"),
("Conflict", "They shifted focus to tone rather than substance"),

("Moral Tradeoffs", "They prioritized personal comfort over helping someone in need"),
("Moral Tradeoffs", "They chose not to intervene in a questionable situation"),
("Moral Tradeoffs", "They justified inaction by assuming someone else would act"),
("Moral Tradeoffs", "They helped selectively based on visibility of the act"),
("Moral Tradeoffs", "They weighed fairness against personal benefit and chose the latter"),
("Moral Tradeoffs", "They ignored minor harm because addressing it was inconvenient"),
("Moral Tradeoffs", "They rationalized benefiting from an unfair system"),
("Moral Tradeoffs", "They chose efficiency over fairness in decision-making"),
("Moral Tradeoffs", "They justified harm as unintended side effects"),
("Moral Tradeoffs", "They deferred ethical decisions to avoid responsibility")
]

df = pd.DataFrame(behaviors, columns=["category", "behavior"])
df

In [ ]:
df.to_csv("./data/morally_grayzone_behaviors.csv", index=False)